# Kinthread r8 (any account)
Starts from r7 (weights in the repo), adds a prefix-LM reader (the question is read both ways), dataset v8 (answer labels and sentence constructions balanced to real exams: `dataset_spec.py`, `data/dataset_v8_manifest.json`) and on-the-fly exam phrasing. Checkpoints are picked and stopped by real dev questions, with health checks. Run all; run all again to resume.

In [ ]:
import os, torch, hashlib
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'
REPO = '/content/Kinthread-llm-model'
if not os.path.exists(REPO):
    !git clone --depth 1 -b main https://github.com/ArAnirudh2901/Kinthread-llm-model.git {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd {REPO}
INIT = 'runs/v10-r7/best.pt'
if not os.path.exists(INIT):
    !cat runs/v10-r7/best.pt.part-* > {INIT}
assert hashlib.sha256(open(INIT, 'rb').read()).hexdigest() == open('runs/v10-r7/best.pt.sha256').read().strip(), 'r7 weights corrupted'
!pip -q install -U gdown
S = '/content/share'   # read-only copies from the project's link-viewable Drive folder
os.makedirs(S, exist_ok=True)
if not os.path.isdir(f'{S}/dataset_v7'):
    !gdown -q --folder https://drive.google.com/drive/folders/1MNiQuramSX5ImytxEdvNEIN7SgFtT6k8 -O {S}/dataset_v7
FILES = {'real_questions.tgz': '1yIanNsRCb6-BAkGIF_SIjp9ZMCRIotGy', 'star_r3.jsonl.gz': '1_qk0JL_lkzKBdv2vwHxWijI4yNrNPRtr',
         'star_r2.jsonl.gz': '1s_RQLHpNbWsttRTjaoFpV1NP5I4q3Veh', 'corrections.jsonl.gz': '1VW_dWGo2GkDdV9AKVWl-pxz-IFPrpEZp'}
for name, fid in FILES.items():
    if not os.path.exists(f'{S}/{name}'):
        !wget -q -O {S}/{name} "https://drive.usercontent.google.com/download?id={fid}&export=download&confirm=t"
os.makedirs('data/real_questions', exist_ok=True)
!tar -xzf {S}/real_questions.tgz -C data/real_questions 2>/dev/null
from google.colab import drive
drive.mount('/content/drive')
BASE = next((b for b in ('/content/drive/MyDrive/kinthread_shared',) if os.path.isdir(b)), '/content/drive/MyDrive/kinthread_r8')
OUT = f'{BASE}/runs/v10-r8'
os.makedirs(OUT, exist_ok=True)
DATA = f'{S}/dataset_v7'
print('init', INIT, '| data', os.listdir(DATA), '| out', OUT)

In [ ]:
# r8: r7 + prefix-LM (question read both ways) + dataset v8 (answers and constructions balanced to real exams)
#     + on-the-fly exam phrasing of pool rows; picked and stopped by all 575 old-dev real questions; health checks
EXTRA = [f'{DATA}/reworded.jsonl.gz', 'data/dataset_v8_extra.jsonl.gz*3', 'data/phrase_balance.jsonl.gz', 'data/phrase_boost.jsonl.gz',
         f'{S}/star_r3.jsonl.gz*3', f'{S}/star_r2.jsonl.gz*3', 'data/teach_rows.jsonl.gz*3', 'data/teach_sg2.jsonl.gz*3', 'data/pairs_v1.jsonl.gz']
EXTRA = ','.join(e for e in EXTRA if os.path.exists(e.split('*')[0]))
print('extra rows:', EXTRA)
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python train_reader.py --arch v10 --run v10-r8 --out {OUT} --pool {DATA} --init {INIT} --prefix_lm --of_form --style_p 0.8 --steps 15000 --batch 64 --lr 3e-4 --warmup 500 --optim muon --amp --workers 2 --resume --eval_every 1500 --save_every 1000 --reworded {EXTRA} --reworded_dev data/reworded_dev.jsonl.gz --p_reworded 0.6 --ramp_steps 2000 --block 448 --real_dev 575 --select_real

In [ ]:
!python -m eval.run_real --ckpt {OUT}/best.pt --split dev --adaptive 1.0 --hybrid --out {OUT}/real_dev.json | grep -E '"dev'
!python -m eval.construction_suite --ckpt {OUT}/best.pt --n 15 --out {OUT}/construction_suite.json | tail -3